# 03 — Cosa impara il classificatore a feature, e cosa manca al transformer

TF-IDF + regressione logistica batte ModernBERT, soprattutto con le pagine del sito (input C). Prima di
rilanciare il transformer (nel weekend, con più configurazioni) vogliamo capire **da dove viene il
vantaggio** e quindi **quali leve provare**. Ogni sezione risponde a una domanda e porta a una scelta
per il rilancio (riassunte nella sezione 20).

**Il focus resta quello del paper: solo testo, livello NewsGuard complessivo.** I 9 criteri NewsGuard
servono qui per *capire* cosa c'è nel testo; usarli per addestrare o valutare è un'analisi in più, da fare
poi anche con il feature.

**Cosa usa il feature**
- **H1 — le pagine dicono i criteri NewsGuard.** Il punteggio NewsGuard è la somma di 9 criteri; alcuni
  (corregge gli errori, dichiara proprietà e fondi, dice chi comanda, firma gli autori) si leggono proprio
  nelle pagine about, ownership, corrections, staff. Abbiamo i 9 criteri per ogni testata: si può verificare.
- **H2 — stile e template di un tipo di sito**, che passano da una rete all'altra.
- **H3 — impronte delle testate**: parole di una sola testata, che dicono quanto il modello memorizza.

**Cosa limita il transformer**
- **T1 — vede meno testo**: 400 parole di articolo e 400 di pagine, il feature legge tutto.
- **T2 — niente pesi di classe**: il feature usa `class_weight="balanced"`, il transformer no.
- **T3 — poche etichette vere**: 45 mila articoli ma 272 etichette, una per testata; il transformer impara
  a memoria le testate (loss di training a 0).
- **T4 — le classi sono ordinate** ($l_1 < \dots < l_5$) ma la loss le tratta come etichette senza ordine.

Tutto sullo schema **per gruppo** (quello principale). Gli addestramenti stanno in
`Results/2-classificazione/03_analisi/` e, se ci sono già, si ricaricano: rilanciare per cambiare un
grafico non riaddestra nulla. Sul server:

```bash
cd ~/John/WebScience2027/Tentativi/02_ScrapingConPagine/2-classificazione
screen -dmS analisi papermill 03_cosa_impara_il_feature.ipynb ../Results/2-classificazione/esecuzioni/03_analisi.ipynb -p ESEGUI_SONDA True
```

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

from comune import CLASSI, SEED, carica, cartella_risultati, letture, per_testata, registratore, taglia

LIVELLI = ["l1", "l2", "l3", "l4", "l5"]

In [ ]:
SCHEMA = "fold_gruppo"           # la valutazione principale
TRANSFORMER = "modernbert_es"    # con cosa confrontare: Results/2-classificazione/<TRANSFORMER>_<A|C>_<SCHEMA>_articoli.csv
K_ABLAZIONE = [0, 100, 1000, 5000]
QUOTE_TESTATE = [25, 50]         # curve di apprendimento: % delle testate di training
ARTICOLI_PER_TESTATA = [5, 10, 20]
N_JOBS = 5                       # fold in parallelo (uno per processo)
ESEGUI_SONDA = False             # sezione 18: serve la GPU
PROVA = False                    # True: 40 testate, risultati in Results/2-classificazione/prova/

In [ ]:
OUT = cartella_risultati(PROVA)
ANALISI = OUT / "03_analisi"
ANALISI.mkdir(exist_ok=True)
registra = registratore(OUT / "avanzamento_analisi.log")
art = carica(PROVA)
testate = art.drop_duplicates("domain").set_index("domain")[["label", "gruppo", SCHEMA, "pagine"]]
art = art.drop(columns="pagine")            # le pagine si attaccano quando servono, per non tenerle ripetute
FOLDS = sorted(art[SCHEMA].unique())

# NewsGuard: punteggio e i 9 criteri (Yes / No)
ng = pd.read_csv("../../../../../Dataset/etichette/newsguard_2022.csv")
ng = ng.assign(domain=ng.Domain.str.lower()).drop_duplicates("domain").set_index("domain")
i = ng.columns.get_loc("Does not repeatedly publish false content")
NOMI_CRITERI = ["niente falsi", "raccolta responsabile", "corregge gli errori", "separa notizie e opinioni",
                "titoli non ingannevoli", "dichiara proprietà e fondi", "segnala la pubblicità",
                "dice chi comanda", "firma gli autori"]
PUNTI = [22, 18, 12.5, 12.5, 10, 7.5, 7.5, 5, 5]     # i pesi dei criteri nel punteggio NewsGuard
CRITERI_PAGINE = ["corregge gli errori", "dichiara proprietà e fondi", "segnala la pubblicità",
                  "dice chi comanda", "firma gli autori"]
criteri = ng.reindex(testate.index).iloc[:, i:i + 9].set_axis(NOMI_CRITERI, axis=1)
testate["score"] = ng.Score.reindex(testate.index)
art["score"] = art.domain.map(testate.score)

# lo stesso input che vede il transformer (02): 400 parole di articolo, 400 di pagine
pag_400 = testate.pagine.map(lambda t: taglia(t, 400))
art["A_400"] = art.testo.map(lambda t: taglia(t, 400))
testi_art = art.groupby("domain").testo.apply("\n".join).reindex(testate.index)   # tutti gli articoli di una testata

# input: (quanto articolo, quanto pagine)
INPUT = {"A": ("intero", None), "C": ("intero", "intero"), "A_400": ("400", None),
         "C_400_400": ("400", "400"), "C_intero_400": ("intero", "400"), "C_400_intero": ("400", "intero")}


def testi(df, variante):
    a, p = INPUT[variante]
    t = df.testo if a == "intero" else df.A_400
    if p:
        t = t + "\n\n" + df.domain.map(testate.pagine if p == "intero" else pag_400)
    return t


PARAM_TFIDF = dict(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                   max_features=100_000, strip_accents="unicode")   # come in 01


def nuova_lr(**kw):
    return LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED, **kw)


def dividi(fold):
    return art[art[SCHEMA] != fold], art[art[SCHEMA] == fold]


def prob(clf, X, te):
    return te.index.to_numpy(), clf.classes_, clf.predict_proba(X)


def adatta(tr, te, variante="C"):
    """TF-IDF + LR come in 01, sul testo della variante."""
    vec = TfidfVectorizer(**PARAM_TFIDF)
    return prob(nuova_lr().fit(vec.fit_transform(testi(tr, variante)), tr.label), vec.transform(testi(te, variante)), te)


def ad_articoli(P, dom_te, te):
    """Predizioni per testata -> a ogni articolo la predizione della sua testata."""
    riga = pd.Series(range(len(dom_te)), index=dom_te)[te.domain].to_numpy()
    return P[riga]


def come_pred(proba):
    pred = pd.DataFrame(proba, columns=CLASSI)
    pred.insert(0, "domain", art.domain.values)
    pred.insert(0, "article_id", art.article_id.values)
    pred["fold"] = art[SCHEMA].values
    pred["y_true"] = art.label.values
    pred["y_pred"] = proba.argmax(1)
    return pred


def file_pred(variante):
    return ANALISI / f"pred_{variante}.csv"


def esegui(nome, funzione, varianti, altri_file=()):
    """Se ci sono già le predizioni di tutte le varianti (e gli altri file) non fa nulla e restituisce None;
    altrimenti esegue funzione(fold) su tutti i fold in parallelo e salva le predizioni."""
    if all(file_pred(v).exists() for v in varianti) and all(f.exists() for f in altri_file):
        registra(f"{nome}: già fatto, carico")
        return None
    t0 = time.time()
    registra(f"{nome}: inizio")
    risultati = Parallel(n_jobs=N_JOBS)(delayed(funzione)(fold) for fold in FOLDS)
    for v in varianti:
        proba = np.zeros((len(art), 5))
        for r in risultati:
            idx, classi, P = r["pred"][v]
            proba[idx[:, None], classi[None, :]] = P
        come_pred(proba).to_csv(file_pred(v), index=False)
    registra(f"{nome}: fatto in {(time.time() - t0) / 60:.0f} min")
    return risultati


registra(f"=== analisi: {len(art)} articoli, {len(testate)} testate, schema {SCHEMA} ===")

# Parte A — Cosa vede ciascun modello

## 1. In parole (T1)

Il feature legge articolo e pagine interi; il transformer le prime 400 parole di ciascuno.

In [ ]:
def quanto(parole):
    return {"mediana parole": parole.median(),
            "% oltre 400 parole": round(100 * (parole > 400).mean(), 1),
            "% delle parole viste dal transformer": round(100 * np.minimum(parole, 400).sum() / parole.sum(), 1)}


pd.DataFrame({"articoli": quanto(art.testo.str.split().str.len()),
              "pagine (una volta per testata)": quanto(testate.pagine.str.split().str.len())})

## 2. In token di ModernBERT: quanto contesto servirebbe

Su un campione di 3000 articoli: quanti token servono per articolo, pagine e articolo + pagine interi, e
quanti articoli ci stanno **per intero** con 1024, 2048, 4096 o 8192 token (ModernBERT arriva a 8192).

In [ ]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    camp = art.sample(min(3000, len(art)), random_state=SEED)
    n_art = pd.Series([len(x) for x in tok(camp.testo.tolist(), add_special_tokens=False).input_ids], index=camp.index)
    n_pag = pd.Series([len(x) for x in tok(testate.pagine.tolist(), add_special_tokens=False).input_ids], index=testate.index)
    n_tot = n_art + camp.domain.map(n_pag).to_numpy()
    token = pd.DataFrame({"articolo": n_art, "pagine": camp.domain.map(n_pag), "articolo + pagine": n_tot})
    display(token.describe(percentiles=[0.5, 0.75, 0.9, 0.95]).round(0))
    display(pd.DataFrame({m: (token <= m).mean().round(2) for m in [512, 1024, 2048, 4096, 8192]}).T
            .rename_axis("max token").add_prefix("ci sta tutto: "))
except ImportError:
    print("transformers non installato qui: sezione saltata")

# Parte B — Gli addestramenti dell'analisi

Tutti con le stesse impostazioni di 01 e sugli stessi fold; i fold girano in parallelo.

| Gruppo | Varianti | Per rispondere a |
|---|---|---|
| due blocchi | `due_blocchi` (come C ma vocabolario separato per articolo e pagine), `solo_pagine`, `senza_top<k>` (tolte le k feature più forti) | da dove viene il segnale; quanto è concentrato |
| tagli | `A_400`, `C_400_400` (= l'input del transformer), `C_intero_400`, `C_400_intero` | quanto conta la lunghezza, e di cosa |
| etichette | `C_senza_pesi`, `regressione_punteggio`, `testate_<q>`, `articoli_<k>` | pesi di classe, ordine delle classi, quante etichette servono |
| testata | `testata_tutto`, `testata_pagine` (un documento per testata) | serve addestrare per articolo? |

Sul server ci vuole all'incirca un'ora o due.

In [ ]:
VARIANTI_BLOCCHI = ["due_blocchi", "solo_pagine"] + [f"senza_top{k}" for k in K_ABLAZIONE if k > 0]
FILE_COEF, FILE_CONTR = ANALISI / "coef_due_blocchi.pkl", ANALISI / "contributi.csv"


def esperimento_blocchi(fold):
    t0 = time.time()
    tr, te = dividi(fold)
    v_art, v_pag = TfidfVectorizer(**PARAM_TFIDF), TfidfVectorizer(**PARAM_TFIDF)
    A_tr, A_te = v_art.fit_transform(tr.testo), v_art.transform(te.testo)
    P_tr, P_te = v_pag.fit_transform(tr.domain.map(testate.pagine)), v_pag.transform(te.domain.map(testate.pagine))
    nomi = np.r_[["art: " + t for t in v_art.get_feature_names_out()],
                 ["pag: " + t for t in v_pag.get_feature_names_out()]]
    X_tr, X_te = hstack([A_tr, P_tr]).tocsr(), hstack([A_te, P_te]).tocsr()
    out = {}
    clf = nuova_lr().fit(X_tr, tr.label)
    out["due_blocchi"] = prob(clf, X_te, te)
    coef = pd.DataFrame(clf.coef_.T, index=nomi, columns=[LIVELLI[c] for c in clf.classes_])
    # margine della classe predetta (rispetto alla media delle classi), diviso tra i due blocchi
    na = A_tr.shape[1]
    s_art, s_pag = A_te @ clf.coef_[:, :na].T, P_te @ clf.coef_[:, na:].T
    c, i = clf.predict(X_te), np.arange(len(te))
    contributi = pd.DataFrame({
        "article_id": te.article_id.values, "domain": te.domain.values, "fold": fold,
        "y_true": te.label.values, "y_pred": c,
        "margine_articolo": s_art[i, c] - s_art.mean(1), "margine_pagine": s_pag[i, c] - s_pag.mean(1)})
    out["solo_pagine"] = prob(nuova_lr().fit(P_tr, tr.label), P_te, te)
    ordine = np.argsort(-np.abs(clf.coef_).max(0))      # scelte sul training del fold
    for k in [k for k in K_ABLAZIONE if k > 0]:
        tieni = np.sort(ordine[k:])
        out[f"senza_top{k}"] = prob(nuova_lr().fit(X_tr[:, tieni], tr.label), X_te[:, tieni], te)
    registra(f"due blocchi fold {fold}: fatto in {time.time() - t0:.0f} s ({na} feature articolo, {P_tr.shape[1]} pagine)")
    return {"pred": out, "fold": fold, "coef": coef, "contributi": contributi}


ris = esegui("due blocchi", esperimento_blocchi, VARIANTI_BLOCCHI, [FILE_COEF, FILE_CONTR])
if ris is not None:
    pd.to_pickle({r["fold"]: r["coef"] for r in ris}, FILE_COEF)
    pd.concat([r["contributi"] for r in ris]).to_csv(FILE_CONTR, index=False)
coef, contributi = pd.read_pickle(FILE_COEF), pd.read_csv(FILE_CONTR)

In [ ]:
VARIANTI_TAGLI = ["A_400", "C_400_400", "C_intero_400", "C_400_intero"]


def esperimento_tagli(fold):
    tr, te = dividi(fold)
    return {"pred": {v: adatta(tr, te, v) for v in VARIANTI_TAGLI}}


esegui("tagli", esperimento_tagli, VARIANTI_TAGLI);

In [ ]:
VARIANTI_ETICHETTE = (["C_senza_pesi", "regressione_punteggio"] + [f"testate_{q}" for q in QUOTE_TESTATE]
                      + [f"articoli_{k}" for k in ARTICOLI_PER_TESTATA])


def soglie(tr):
    """Soglie tra livelli sul punteggio: a metà tra il massimo di un livello e il minimo del successivo (training)."""
    s = tr.groupby("label").score
    return [(s.max()[k] + s.min()[k + 1]) / 2 for k in range(4)]


def esperimento_etichette(fold):
    t0 = time.time()
    tr, te = dividi(fold)
    out = {}
    vec = TfidfVectorizer(**PARAM_TFIDF)
    X_tr, X_te = vec.fit_transform(testi(tr, "C")), vec.transform(testi(te, "C"))
    out["C_senza_pesi"] = prob(LogisticRegression(max_iter=1000, random_state=SEED).fit(X_tr, tr.label), X_te, te)
    livello = np.digitize(Ridge().fit(X_tr, tr.score).predict(X_te), soglie(tr))
    out["regressione_punteggio"] = (te.index.to_numpy(), np.arange(5), np.eye(5)[livello])
    dom = testate[testate[SCHEMA] != fold]
    for q in QUOTE_TESTATE:
        scelte = dom.groupby("label").sample(frac=q / 100, random_state=SEED).index
        out[f"testate_{q}"] = adatta(tr[tr.domain.isin(scelte)], te)
    mescolato = tr.sample(frac=1, random_state=SEED)
    for k in ARTICOLI_PER_TESTATA:
        out[f"articoli_{k}"] = adatta(mescolato.groupby("domain").head(k), te)
    registra(f"etichette fold {fold}: fatto in {time.time() - t0:.0f} s")
    return {"pred": out}


esegui("etichette", esperimento_etichette, VARIANTI_ETICHETTE);

In [ ]:
VARIANTI_TESTATA = ["testata_tutto", "testata_pagine"]
doc_testata = {"testata_tutto": testi_art + "\n\n" + testate.pagine, "testata_pagine": testate.pagine}


def esperimento_testata(fold):
    _, te = dividi(fold)
    dom_tr, dom_te = testate.index[testate[SCHEMA] != fold], testate.index[testate[SCHEMA] == fold]
    out = {}
    for v, docs in doc_testata.items():
        vec = TfidfVectorizer(**PARAM_TFIDF)
        clf = nuova_lr().fit(vec.fit_transform(docs[dom_tr]), testate.label[dom_tr])
        out[v] = (te.index.to_numpy(), clf.classes_, ad_articoli(clf.predict_proba(vec.transform(docs[dom_te])), dom_te, te))
    return {"pred": out}


esegui("una riga per testata", esperimento_testata, VARIANTI_TESTATA);

## 3. Tutto in una tabella

Le righe di 01 e dei transformer si leggono dai loro file (se mancano si saltano). Per testata = voto di
maggioranza; binario = soglia tra $l_2$ e $l_3$. Per le varianti `testata_*` contano solo le colonne per
testata.

In [ ]:
pred = {v: pd.read_csv(file_pred(v)) for v in VARIANTI_BLOCCHI + VARIANTI_TAGLI + VARIANTI_ETICHETTE + VARIANTI_TESTATA}
righe = {}
for etichetta, f in [("feature A (01)", OUT / f"feature_A_{SCHEMA}_articoli.csv"),
                     ("feature C (01)", OUT / f"feature_C_{SCHEMA}_articoli.csv")] + [
                    (f"{t} {c}", OUT / f"{t}_{c}_{SCHEMA}_articoli.csv")
                    for t in ["modernbert", "modernbert_es"] for c in ["A", "C"]]:
    if f.exists():
        righe[etichetta] = letture(pd.read_csv(f))
    else:
        print("manca", f.name)
for v, p in pred.items():
    righe[f"feature {v}"] = letture(p)
tabella = pd.DataFrame(righe).T
tabella.to_csv(ANALISI / "tabella.csv")
COLONNE = ["testata 5cl", "testata F1", "testata bin", "testata tol0.5", "articolo 5cl", "articolo bin"]
tabella[COLONNE]

# Parte C — Cosa impara il classificatore a feature

## 4. Da dove viene il segnale: articolo o pagine?

`due_blocchi` deve andare circa come C (tabella sopra), altrimenti la divisione non dice nulla su C. Per ogni
articolo del test, il margine della classe predetta si divide in una parte dovuta alle parole
dell'articolo e una dovuta alle parole delle pagine: `quota pagine` = |pagine| / (|articolo| + |pagine|).

In [ ]:
contributi["quota_pagine"] = contributi.margine_pagine.abs() / (
    contributi.margine_articolo.abs() + contributi.margine_pagine.abs())
contributi["giusto"] = contributi.y_true == contributi.y_pred
print(f"quota pagine, mediana su tutti gli articoli: {contributi.quota_pagine.median():.2f}")
print(f"le pagine spingono verso la classe predetta nel {100 * (contributi.margine_pagine > 0).mean():.0f}% "
      f"degli articoli, l'articolo nel {100 * (contributi.margine_articolo > 0).mean():.0f}%")
display(contributi.groupby(contributi.y_true.map(dict(enumerate(LIVELLI))))
        .agg(quota_pagine=("quota_pagine", "median"), accuratezza=("giusto", "mean")).round(2).T)
display(tabella.loc[[r for r in ["feature A (01)", "feature C (01)", "feature due_blocchi", "feature solo_pagine",
                                 "feature testata_pagine"] if r in tabella.index], COLONNE])

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist([contributi.quota_pagine[contributi.giusto], contributi.quota_pagine[~contributi.giusto]],
        bins=20, label=["predizione giusta", "sbagliata"], stacked=True)
ax.set_xlabel("quota del margine dovuta alle pagine"); ax.set_ylabel("articoli"); ax.legend()
plt.tight_layout(); plt.show()

## 5. Le parole che contano

Coefficiente medio sui fold (una parola assente dal vocabolario di un fold conta 0). Positivo = spinge
verso quella classe rispetto alle altre. `fold` = in quanti fold la parola è tra le prime 100 della classe:
5 vuol dire stabile.

In [ ]:
tutti = pd.concat(coef, axis=1).fillna(0)                      # colonne: (fold, livello)
medio = tutti.T.groupby(level=1).mean().T[LIVELLI]


def nei_top(liv, n=100):
    conta = pd.Series(0, index=medio.index)
    for c in coef.values():
        conta[c[liv].nlargest(n).index] += 1
    return conta


def top(liv, n=25):
    t = medio[liv].nlargest(n).to_frame("coef")
    t["fold"] = nei_top(liv)[t.index]
    return t.round(2)


pd.concat({liv: top(liv).reset_index().rename(columns={"index": "feature"}) for liv in ["l1", "l3", "l5"]}, axis=1)

In [ ]:
fig, assi = plt.subplots(1, 5, figsize=(20, 6))
for ax, liv in zip(assi, LIVELLI):
    t = medio[liv].nlargest(20)[::-1]
    ax.barh(range(len(t)), t.values, color=["C1" if s.startswith("pag") else "C0" for s in t.index])
    ax.set_yticks(range(len(t))); ax.set_yticklabels([s[5:] for s in t.index], fontsize=8)
    ax.set_title(f"verso {liv}")
fig.suptitle("Le 20 feature più forti per classe (blu: articolo, arancio: pagine)")
plt.tight_layout(); plt.show()

pd.concat({liv: top(liv, 200) for liv in LIVELLI}).to_csv(ANALISI / "top_feature.csv")
print("quota di feature delle pagine tra le più forti:")
pd.DataFrame({liv: {f"top {n}": medio[liv].nlargest(n).index.str.startswith("pag").mean()
                    for n in [20, 100, 1000]} for liv in LIVELLI}).round(2)

## 6. Impronta di una testata o segnale generale? (H2 contro H3)

Per le 100 feature più forti di ogni classe: in **quante testate** compaiono (negli articoli o nelle
pagine, secondo il blocco) e quante di quelle testate sono davvero della classe (`purezza`). In 1–3
testate è un'impronta; in tante testate della stessa classe è un segnale che può passare a testate nuove.

In [ ]:
forti = {liv: medio[liv].nlargest(100).index for liv in LIVELLI}
termini = sorted(set().union(*forti.values()))


def presenza(blocco, docs):
    voc = sorted({t[5:] for t in termini if t.startswith(blocco)})
    cv = CountVectorizer(vocabulary=voc, ngram_range=(1, 2), strip_accents="unicode", binary=True)
    return pd.DataFrame(cv.fit_transform(docs).toarray(), index=docs.index, columns=[blocco + t for t in voc])


pres = pd.concat([presenza("art: ", testi_art), presenza("pag: ", testate.pagine)], axis=1)
n_testate = pres.sum()


def purezza(t, i):
    return (testate.label[pres[t] == 1] == i).mean()


righe = []
for i, liv in enumerate(LIVELLI):
    for blocco in ["art: ", "pag: "]:
        f = [t for t in forti[liv] if t.startswith(blocco)]
        if f:
            righe.append({"classe": liv, "blocco": blocco[:3], "feature": len(f),
                          "mediana testate": n_testate[f].median(),
                          "% impronte (≤3 testate)": round(100 * (n_testate[f] <= 3).mean()),
                          "purezza media": round(np.mean([purezza(t, i) for t in f]), 2)})
display(pd.DataFrame(righe).set_index(["classe", "blocco"]))


def esempi(liv, n=15):
    i, f = LIVELLI.index(liv), list(forti[liv][:n])
    return pd.DataFrame({"feature": f, "coef": medio.loc[f, liv].round(2).values, "testate": n_testate[f].values,
                         "purezza": [round(purezza(t, i), 2) for t in f]})


pd.concat({liv: esempi(liv) for liv in ["l1", "l5"]}, axis=1)

## 7. I criteri NewsGuard (H1)

**a. Cosa separa un livello dal successivo.** Quota di testate di ogni livello che soddisfa ciascun
criterio, e per ogni salto di livello i criteri che cambiano di più. `*` = criterio che si legge nelle
pagine del sito.

In [ ]:
si = criteri.eq("Yes")
per_livello = si.groupby(testate.label.map(dict(enumerate(LIVELLI)))).mean()
per_livello.columns = [c + (" *" if c in CRITERI_PAGINE else "") for c in per_livello.columns]
display(per_livello.T.round(2))

salti = per_livello.diff().iloc[1:]
display(pd.DataFrame({f"{LIVELLI[k]} → {LIVELLI[k + 1]}": salti.iloc[k].nlargest(3).round(2).reset_index()
                      .apply(lambda r: f"{r.iloc[0]} (+{r.iloc[1]:.2f})", axis=1).values for k in range(4)}))

# controllo: il punteggio è la somma dei punti dei criteri soddisfatti?
punti = (si * PUNTI).sum(axis=1)
print(f"punteggio = somma dei punti per il {100 * (punti == testate.score).mean():.0f}% delle testate "
      f"({criteri.isna().any(axis=1).sum()} testate hanno almeno un criterio vuoto)")

**b. Quali criteri si riconoscono dal testo, e da quale testo.** Per ogni criterio, una regressione
logistica per testata (un documento per testata, stessi fold per gruppo) dalle **pagine** o da **tutti gli
articoli** della testata. Accuratezza bilanciata: 0.5 = a caso. Se i criteri delle pagine si riconoscono
bene dalle pagine, si potrà dare al transformer i criteri come etichette in più (multi-task): è
un'analisi in più rispetto al focus del paper, e andrebbe fatta anche per il feature.

In [ ]:
FILE_CRITERI = ANALISI / "criteri_predetti.csv"
if FILE_CRITERI.exists():
    oof = pd.read_csv(FILE_CRITERI, index_col=[0, 1])
else:
    y_crit = criteri.apply(lambda s: s.map({"Yes": 1.0, "No": 0.0}))
    oof = {}
    for fold in FOLDS:
        dom_tr, dom_te = testate.index[testate[SCHEMA] != fold], testate.index[testate[SCHEMA] == fold]
        for fonte, docs in [("pagine", testate.pagine), ("articoli", testi_art)]:
            vec = TfidfVectorizer(**PARAM_TFIDF)
            X_tr, X_te = vec.fit_transform(docs[dom_tr]), vec.transform(docs[dom_te])
            for c in NOMI_CRITERI:
                y = y_crit.loc[dom_tr, c]
                ok = y.notna().to_numpy()
                if y[ok].nunique() < 2:
                    continue
                p = nuova_lr().fit(X_tr[ok], y[ok]).predict_proba(X_te)[:, 1]
                for d, v in zip(dom_te, p):
                    oof[(fonte, d, c)] = v
        registra(f"criteri fold {fold}: fatto")
    oof = pd.Series(oof).unstack()
    oof.index.names = ["fonte", "domain"]
    oof.to_csv(FILE_CRITERI)

righe = {}
for fonte in ["pagine", "articoli"]:
    for c in NOMI_CRITERI:
        vero = criteri[c].map({"Yes": 1, "No": 0})
        p = oof.loc[fonte, c].reindex(testate.index)
        ok = vero.notna() & p.notna()
        righe.setdefault(c + (" *" if c in CRITERI_PAGINE else ""), {})[fonte] = balanced_accuracy_score(vero[ok], p[ok] > 0.5)
pd.DataFrame(righe).T.round(2).rename(columns=lambda f: f"acc. bilanciata da {f}")

# Parte D — Perché il transformer resta indietro, e cosa provare

## 8. Lunghezza del testo e parole più forti (T1)

`C_400_400` è il feature sullo stesso testo che vede il transformer. Se resta vicino a C, la lunghezza non
spiega il distacco; se scende verso il transformer, sì. `C_intero_400` e `C_400_intero` dicono se il testo
che manca serve di più nell'articolo o nelle pagine (dove spendere i token in più).

Ablazione: togliendo le k feature più forti, se l'accuratezza regge il segnale è sparso su tante parole.

In [ ]:
display(tabella.loc[[r for r in ["feature C (01)", "feature C_intero_400", "feature C_400_intero", "feature C_400_400",
                                 "feature A (01)", "feature A_400", f"{TRANSFORMER} A", f"{TRANSFORMER} C"]
                     if r in tabella.index], COLONNE])

abl = pd.DataFrame({k: letture(pred["due_blocchi" if k == 0 else f"senza_top{k}"]) for k in K_ABLAZIONE}).T
fig, ax = plt.subplots(figsize=(5, 3))
for m in ["testata 5cl", "testata bin", "articolo 5cl"]:
    ax.plot(range(len(K_ABLAZIONE)), abl[m], marker="o", label=m)
ax.set_xticks(range(len(K_ABLAZIONE))); ax.set_xticklabels(K_ABLAZIONE)
ax.set_xlabel("feature più forti tolte"); ax.set_ylabel("accuratezza"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Pesi di classe (T2)

Il feature senza `class_weight="balanced"`: quanto perde, e su quali classi. Le classi estreme
($l_1$, $l_5$) sono le meno numerose.

In [ ]:
def acc_per_classe(p):
    t = per_testata(p)
    return (t.y_true == t.y_pred).groupby(t.y_true).mean().rename(index=dict(enumerate(LIVELLI)))


display(tabella.loc[["feature C (01)", "feature C_senza_pesi"], COLONNE])
pd.DataFrame({"testate": testate.label.value_counts().sort_index().rename(index=dict(enumerate(LIVELLI))),
              "con pesi": acc_per_classe(pd.read_csv(OUT / f"feature_C_{SCHEMA}_articoli.csv")),
              "senza pesi": acc_per_classe(pred["C_senza_pesi"])}).round(2)

## 10. Quante etichette servono (T3)

Il feature addestrato su una parte delle testate di training, o su pochi articoli per testata. Se con 5–10
articoli per testata va come con tutti, gli articoli in più sono ripetizioni della stessa etichetta: per il
transformer conviene **pescare pochi articoli per testata a ogni epoca** (meno memorizzazione, epoche più
veloci). Se la curva sulle testate sale ancora al 100%, il limite sono le etichette, non il modello.

In [ ]:
curva_t = pd.DataFrame({f"{q}%": letture(pred[f"testate_{q}"]) for q in QUOTE_TESTATE} | {"100%": tabella.loc["feature C (01)"]}).T
curva_a = pd.DataFrame({str(k): letture(pred[f"articoli_{k}"]) for k in ARTICOLI_PER_TESTATA} | {"tutti": tabella.loc["feature C (01)"]}).T
fig, assi = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, curva, titolo in [(assi[0], curva_t, "testate di training usate"), (assi[1], curva_a, "articoli per testata")]:
    for m in ["testata 5cl", "testata bin", "articolo 5cl"]:
        ax.plot(curva.index, curva[m], marker="o", label=m)
    ax.set_xlabel(titolo)
assi[0].set_ylabel("accuratezza"); assi[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 11. Serve addestrare per articolo?

`testata_tutto`: un solo documento per testata (tutti gli articoli + le pagine), 217 esempi di training per
fold. Se va come C, l'informazione utile sta nella testata nel suo insieme: per il transformer ha senso
un modello che legge **più articoli insieme** e decide per la testata (multiple instance learning), invece
di classificare ogni articolo da solo.

In [ ]:
tabella.loc[[r for r in ["feature C (01)", "feature testata_tutto", "feature solo_pagine", "feature testata_pagine"]
             if r in tabella.index], [c for c in COLONNE if c.startswith("testata")]]

## 12. Gli errori sono tra livelli vicini? (T4)

Matrici di confusione per testata e quota di errori di un solo livello. Se quasi tutti gli errori sono
adiacenti e la **regressione sul punteggio** NewsGuard va come la classificazione, per il transformer ha
senso una loss che sa dell'ordine (regressione sul punteggio, o ordinale).

In [ ]:
modelli = {"feature C": OUT / f"feature_C_{SCHEMA}_articoli.csv", f"{TRANSFORMER} C": OUT / f"{TRANSFORMER}_C_{SCHEMA}_articoli.csv",
           "regressione punteggio": file_pred("regressione_punteggio")}
modelli = {k: per_testata(pd.read_csv(f)) for k, f in modelli.items() if f.exists()}
fig, assi = plt.subplots(1, len(modelli), figsize=(4.2 * len(modelli), 3.6), squeeze=False)
for ax, (nome, t) in zip(assi[0], modelli.items()):
    m = pd.crosstab(t.y_true, t.y_pred).reindex(index=range(5), columns=range(5), fill_value=0)
    ax.imshow(m, cmap="Blues")
    for (r, c), v in np.ndenumerate(m.to_numpy()):
        ax.text(c, r, v, ha="center", va="center", fontsize=8)
    ax.set_xticks(range(5)); ax.set_xticklabels(LIVELLI); ax.set_yticks(range(5)); ax.set_yticklabels(LIVELLI)
    errori = t[t.y_true != t.y_pred]
    ax.set_title(f"{nome}\nerrori adiacenti: {100 * ((errori.y_true - errori.y_pred).abs() == 1).mean():.0f}%", fontsize=9)
    ax.set_xlabel("predetto"); ax.set_ylabel("vero")
plt.tight_layout(); plt.show()

## 13. Il transformer impara a memoria? (T3)

Loss di training (e di validation, con l'early stopping) per epoca, media sui fold. Una loss di training
vicina a 0 con una validation che sale = memorizza le testate di training.

In [ ]:
for nome in ["modernbert", "modernbert_es"]:
    f = OUT / f"{nome}_loss.csv"
    if not f.exists():
        print("manca", f.name); continue
    l = pd.read_csv(f).drop_duplicates(["config", "schema", "fold", "epoca"], keep="last")
    l = l[l.schema == SCHEMA]
    print(nome)
    display(l.groupby(["config", "epoca"])[["loss", "val_loss"]].mean().unstack("config").round(3))

## 14. Come si passa dagli articoli alla testata

Il voto di maggioranza butta via le probabilità. Alternative senza parametri: media delle probabilità,
media dei log (prodotto delle probabilità), livello atteso (media di $\sum_k k\,p_k$, arrotondata: tiene
conto dell'ordine). Se un'aggregazione aiuta il transformer più del feature, va usata per entrambi.

In [ ]:
def aggrega(p, modo):
    if modo == "voto":
        return per_testata(p)
    P = p[CLASSI].to_numpy()
    vero = p.groupby("domain").y_true.first()
    if modo == "livello atteso":
        y = np.rint(pd.Series(P @ np.arange(5)).groupby(p.domain.values).mean()).astype(int)
    else:
        y = pd.DataFrame(np.log(P + 1e-9) if modo == "media log" else P).groupby(p.domain.values).mean().idxmax(axis=1)
    return pd.DataFrame({"y_true": vero, "y_pred": y.reindex(vero.index)})


def tre(t):
    d = (t.y_true - t.y_pred).abs()
    return {"5cl": (d == 0).mean(), "bin": ((t.y_true >= 2) == (t.y_pred >= 2)).mean(), "tol0.5": np.where(d == 0, 1, np.where(d == 1, 0.5, 0)).mean()}


righe = {}
for nome, f in [("feature C", OUT / f"feature_C_{SCHEMA}_articoli.csv"), ("feature A", OUT / f"feature_A_{SCHEMA}_articoli.csv"),
                (f"{TRANSFORMER} C", OUT / f"{TRANSFORMER}_C_{SCHEMA}_articoli.csv"), (f"{TRANSFORMER} A", OUT / f"{TRANSFORMER}_A_{SCHEMA}_articoli.csv")]:
    if f.exists():
        p = pd.read_csv(f)
        for modo in ["voto", "media probabilità", "media log", "livello atteso"]:
            for m, v in tre(aggrega(p, modo)).items():
                righe.setdefault((nome, m), {})[modo] = v
pd.DataFrame(righe).T.round(3)

## 15. Feature contro transformer: sbagliano le stesse testate?

Per testata, input C. Se sbagliano testate diverse, combinarli ha senso: come controllo, la **media delle
probabilità** per articolo dei due modelli (nessun parametro, nessun addestramento).

In [ ]:
f_tr, f_fe = OUT / f"{TRANSFORMER}_C_{SCHEMA}_articoli.csv", OUT / f"feature_C_{SCHEMA}_articoli.csv"
if not (f_tr.exists() and f_fe.exists()):
    print("mancano le predizioni di 01 o del transformer: salto")
else:
    fe, tr_ = pd.read_csv(f_fe), pd.read_csv(f_tr)
    a = per_testata(fe)
    b = per_testata(tr_).reindex(a.index)
    display(pd.crosstab((a.y_pred == a.y_true).map({True: "feature giusto", False: "feature sbaglia"}),
                        (b.y_pred == b.y_true).map({True: "transformer giusto", False: "transformer sbaglia"})))
    display(pd.DataFrame({"feature": (a.y_pred == a.y_true).groupby(a.y_true).mean(),
                          "transformer": (b.y_pred == b.y_true).groupby(b.y_true).mean(),
                          "testate": a.y_true.value_counts()}).rename(index=dict(enumerate(LIVELLI))).round(2))
    m = fe.merge(tr_[["article_id"] + CLASSI], on="article_id", suffixes=("_fe", "_tr"))
    P = (m[[c + "_fe" for c in CLASSI]].to_numpy() + m[[c + "_tr" for c in CLASSI]].to_numpy()) / 2
    media = pd.DataFrame({"domain": m.domain, "y_true": m.y_true, "y_pred": P.argmax(1)})
    display(pd.DataFrame({"feature C": letture(fe), "transformer C": letture(tr_), "media dei due": letture(media)}).T[COLONNE])

## 16. Quanto sono sicure le differenze

Con 272 testate (e testate dello stesso gruppo che sbagliano insieme) l'accuratezza per testata ha un
margine d'errore ampio. Bootstrap **per gruppo** (2000 ricampionamenti): intervallo al 95% di ciascun
modello e della differenza tra feature C e transformer C. Dice anche di quanto deve migliorare il
transformer perché il sorpasso sia credibile.

In [ ]:
candidati = {"feature C": OUT / f"feature_C_{SCHEMA}_articoli.csv", "feature A": OUT / f"feature_A_{SCHEMA}_articoli.csv",
             f"{TRANSFORMER} C": OUT / f"{TRANSFORMER}_C_{SCHEMA}_articoli.csv", f"{TRANSFORMER} A": OUT / f"{TRANSFORMER}_A_{SCHEMA}_articoli.csv"}
giusti = pd.DataFrame({k: (lambda t: (t.y_true == t.y_pred).astype(int))(per_testata(pd.read_csv(f)))
                       for k, f in candidati.items() if f.exists()})
gr = testate.gruppo.reindex(giusti.index)
somme, conte = giusti.groupby(gr).sum(), giusti.groupby(gr).size().to_numpy()
idx = np.random.default_rng(SEED).integers(0, len(somme), (2000, len(somme)))
boot = {k: somme[k].to_numpy()[idx].sum(1) / conte[idx].sum(1) for k in giusti}
display(pd.DataFrame({k: {"acc testata 5cl": giusti[k].mean(), "2.5%": np.percentile(b, 2.5), "97.5%": np.percentile(b, 97.5)}
                      for k, b in boot.items()}).T.round(3))
if "feature C" in boot and f"{TRANSFORMER} C" in boot:
    d = boot["feature C"] - boot[f"{TRANSFORMER} C"]
    print(f"feature C - {TRANSFORMER} C: {giusti['feature C'].mean() - giusti[f'{TRANSFORMER} C'].mean():+.3f}, "
          f"intervallo 95% [{np.percentile(d, 2.5):+.3f}, {np.percentile(d, 97.5):+.3f}], "
          f"il feature è avanti nel {100 * (d > 0).mean():.0f}% dei ricampionamenti")

## 17. Quanto costa un addestramento del transformer

Dai log del server: minuti per epoca per modello e input. Serve per decidere quante configurazioni stanno
nel weekend. Il costo cresce circa linearmente con i token (raddoppiare `MAX_TOKEN` ≈ raddoppiare i tempi,
anche di più per la memoria).

In [ ]:
righe = []
for f in OUT.glob("avanzamento_*.log"):
    for riga in open(f):
        m = re.search(r"  (\w+) (fold_\w+) fold (\d+): epoca (\d+)/(\d+) in (\d+) s", riga)
        if m:
            righe.append({"modello": f.stem.replace("avanzamento_", ""), "config": m[1], "schema": m[2], "secondi": int(m[6])})
if righe:
    tempi = pd.DataFrame(righe)
    t = tempi.groupby(["modello", "config"]).secondi.median().div(60).rename("minuti per epoca").to_frame()
    t["ore per 5 fold × 5 epoche"] = (t["minuti per epoca"] * 25 / 60).round(1)
    display(t.round(1))
else:
    print("nessun log di transformer in", OUT)

## 18. Sonda lineare su ModernBERT congelato (T3) — serve la GPU

ModernBERT **senza fine-tuning**: media degli stati dell'ultimo strato (stesso input tagliato del
transformer), e sopra la stessa regressione logistica del feature, per articolo e per testata (media degli
embedding degli articoli). Se la sonda va come la TF-IDF, la rappresentazione basta e il problema è il
fine-tuning; se va peggio, a ModernBERT mancano proprio i segnali lessicali.

In [ ]:
if ESEGUI_SONDA:
    import torch
    from transformers import AutoModel, AutoTokenizer

    righe_sonda = {}
    for config, variante in [("A", "A_400"), ("C", "C_400_400")]:
        f_emb = ANALISI / f"embedding_modernbert_{config}.npy"
        if not f_emb.exists():
            tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
            mod = AutoModel.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False).cuda().half().eval()
            testo, emb = testi(art, variante).tolist(), []
            with torch.no_grad():
                for i in range(0, len(testo), 32):
                    enc = tok(testo[i:i + 32], truncation=True, max_length=1024, padding=True, return_tensors="pt").to("cuda")
                    h, m = mod(**enc).last_hidden_state, enc.attention_mask.unsqueeze(-1)
                    emb.append(((h * m).sum(1) / m.sum(1)).float().cpu().numpy())
            np.save(f_emb, np.vstack(emb))
            del mod; torch.cuda.empty_cache()
            registra(f"embedding {config}: salvati")
        E = np.load(f_emb)
        E_t = pd.DataFrame(E).groupby(art.domain.values).mean().reindex(testate.index).to_numpy()
        p_art, p_tes = np.zeros((len(art), 5)), np.zeros((len(art), 5))
        for fold in FOLDS:
            tr, te = dividi(fold)
            sc = StandardScaler().fit(E[tr.index])
            idx, cl, P = prob(nuova_lr().fit(sc.transform(E[tr.index]), tr.label), sc.transform(E[te.index]), te)
            p_art[idx[:, None], cl[None, :]] = P
            in_tr, in_te = (testate[SCHEMA] != fold).to_numpy(), (testate[SCHEMA] == fold).to_numpy()
            sc = StandardScaler().fit(E_t[in_tr])
            clf = nuova_lr().fit(sc.transform(E_t[in_tr]), testate.label[in_tr])
            p_tes[te.index.to_numpy()[:, None], clf.classes_[None, :]] = ad_articoli(
                clf.predict_proba(sc.transform(E_t[in_te])), testate.index[in_te], te)
        for nome, p in [(f"sonda_{config}", p_art), (f"sonda_testata_{config}", p_tes)]:
            come_pred(p).to_csv(file_pred(nome), index=False)
            righe_sonda[f"feature {nome}"] = letture(come_pred(p))
    tabella = pd.concat([tabella, pd.DataFrame(righe_sonda).T])
    tabella.to_csv(ANALISI / "tabella.csv")
    display(tabella.loc[[r for r in ["feature A_400", "feature C_400_400", f"{TRANSFORMER} A", f"{TRANSFORMER} C"] if r in tabella.index]
                        + list(righe_sonda), COLONNE])
else:
    print("sonda non eseguita (ESEGUI_SONDA = False)")

# Parte E — Dalle risposte alle configurazioni del weekend

## 19. Cosa abbiamo visto

*Da compilare dopo l'esecuzione, sezione per sezione.*

## 20. Le leve, e quando usarle

Ogni leva è una configurazione del transformer da lanciare nel weekend; si scelgono quelle che le
sezioni sopra sostengono, nei limiti di tempo della sezione 17. Le prime sono il focus del paper (solo
testo, livello complessivo); quelle in fondo sono analisi in più.

| Leva | Configurazione | La sosteniamo se… |
|---|---|---|
| **pesi di classe** (T2) | `class_weight` nella cross-entropy | sempre (base di tutte le altre); sez. 9 dice quanto vale |
| **più contesto** (T1) | `MAX_TOKEN` 2048–4096, più parole di articolo e/o pagine | `C_400_400` ben sotto C (sez. 8); i token della sez. 2 dicono quanti ne servono; `C_intero_400` vs `C_400_intero` dice dove |
| **ordine delle classi** (T4) | regressione sul punteggio NewsGuard, o loss ordinale | errori soprattutto adiacenti e `regressione_punteggio` ≥ classificazione (sez. 12) |
| **meno memorizzazione** (T3) | k articoli per testata pescati a ogni epoca, learning rate più basso, weight decay, early stopping per gruppo | curva sugli articoli piatta (sez. 10); loss di training a 0 (sez. 13) |
| **decidere per testata** | più articoli della stessa testata insieme, aggregati (MIL) | `testata_tutto` ≈ C (sez. 11) |
| **pagine a parte** | un encoder per le pagine (una volta per testata) e uno per gli articoli, combinati alla fine | quota pagine alta e `solo_pagine` vicino a C (sez. 4) |
| **aggregazione** | media probabilità o livello atteso al posto del voto | sez. 14 |
| **rappresentazione o fine-tuning?** | testa lineare su ModernBERT congelato / strati congelati | sonda ≈ feature (sez. 18) |
| *in più:* **criteri NewsGuard come etichette** (H1, T3) | multi-task: la testa del livello + 9 teste binarie dei criteri (e lo stesso per il feature) | i criteri delle pagine si riconoscono dalle pagine (sez. 7b) e separano i livelli (sez. 7a) |
| *in più:* **fusione con il feature** | media delle probabilità transformer + TF-IDF | errori su testate diverse, media > entrambi (sez. 15) |

Qualsiasi miglioramento va letto con l'intervallo della sezione 16.